In [1]:
# ============================================
# CS223A Lecture 3 — Euler Angles & Quaternions
# Day 3 — June 21, 2026
# ============================================

import numpy as np

# Yesterday i built rotation matrices manually
# Today i'll try to implement Euler angles and quaternions
# and see exactly WHY quaternions are better

print("Day 3 - Eluer Angels & Quaternions")
print("="*40)

Day 3 - Eluer Angels & Quaternions


In [4]:
# --- Euler Angles (Z-Y-X convention) ---
# Same as Roll-Pitch-Yaw used in drones, planes, robots
# Three rotations: first Z (yaw), then Y (pitch), then X 

def Rx(angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[1,0,0],[0,c,-s],[0,s,c]])

def Ry(angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[c,0,s],[0,1,0],[-s,0,c]])

def Rz(angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[c,-s,0],[s,c,0],[0,0,1]])

# Z-Y-X Euler angles (yaw, pitch, roll)
yaw = np.radians(30)
pitch = np.radians(45)
roll = np.radians(60)

# Chain them — Z first, then Y, then X
R = Rz(yaw) @ Ry(pitch) @ Rx(roll)

print("Z-Y-X Euler Rotation Matrix:")
print(np.round(R, 3))
print("\nMatrix shape:", R.shape)
print("Determinant (should be 1.0):", round(np.linalg.det(R), 3))

Z-Y-X Euler Rotation Matrix:
[[ 0.612  0.28   0.739]
 [ 0.354  0.739 -0.573]
 [-0.707  0.612  0.354]]

Matrix shape: (3, 3)
Determinant (should be 1.0): 1.0


In [6]:
# --- Demonstrating Gimbal Lock (Singularity) ---
# When pitch = 90 degrees, Z and X axes align
# We lose one degree of freedom

pitch_singular = np.radians(90) # the  problematic angel

R_singular = Rz(yaw) @ Ry(pitch_singular) @ Rx(roll)

print("Rotation Matrix at pitch=90° (singular configuration):")
print(np.round(R_singular, 3))

# Now try to recover the original angles from this matrix
# Using the Z-Y-X formula to extract angles back

beta = np.arctan2(-R_singular[2,0], np.sqrt(R_singular[0,0]**2 + R_singular[1,0]**2))
print("\nRecovered pitch (beta):", np.degrees(beta))

# When cos(beta) = 0, these calculations break down
cos_beta = np.cos(beta)
print("cos(beta):", round(cos_beta, 6))
print("\nWhen cos(beta)=0, we can't distinguish yaw from roll!")
print("This is Gimbal Lock — exactly what Khatib warned about.")

Rotation Matrix at pitch=90° (singular configuration):
[[ 0.     0.5    0.866]
 [ 0.     0.866 -0.5  ]
 [-1.     0.     0.   ]]

Recovered pitch (beta): 90.0
cos(beta): 0.0

When cos(beta)=0, we can't distinguish yaw from roll!
This is Gimbal Lock — exactly what Khatib warned about.


In [12]:
# --- Quaternions ---
# 4 parameters (w, x, y, z) — no singularities ever
# w = cos(theta/2)
# x, y, z = sin(theta/2) * rotation_axis

def rotation_to_quaternion(R):
    trace = R[0,0] + R[1,1] + R[2,2]

    w = 0.5 * np.sqrt(max(0,1+ trace))
    x = 0.5 * np.sqrt(max(0,1 + R[0,0] - R[1,1] - R[2,2]))
    y = 0.5 * np.sqrt(max(0,1 - R[0,0] + R[1,1] - R[2,2]))
    z = 0.5 * np.sqrt(max(0,1 - R[0,0] - R[1,1] + R[2,2]))

    return np.array([w,x,y,z])

# Convert our normal rotation to quaternion
q_normal = rotation_to_quaternion(R)
print("Normal rotation as quaternion [w,x,y,z]:")
print(np.round(q_normal, 4))
print("Magnitude (should be 1.0):", round(np.linalg.norm(q_normal), 4))

# Convert singular rotation to quaternion
q_singular = rotation_to_quaternion(R_singular)
print("\nSingular rotation (pitch=90°) as quaternion [w,x,y,z]:")
print(np.round(q_singular, 4))
print("Magnitude (should be 1.0):", round(np.linalg.norm(q_singular), 4))
print("\nNo division by zero — quaternions handle it perfectly!")

Normal rotation as quaternion [w,x,y,z]:
[0.8224 0.3604 0.4397 0.0223]
Magnitude (should be 1.0): 1.0

Singular rotation (pitch=90°) as quaternion [w,x,y,z]:
[0.683 0.183 0.683 0.183]
Magnitude (should be 1.0): 1.0

No division by zero — quaternions handle it perfectly!


In [14]:
# --- Summary: Why Quaternions win ---
print("ORIENTATION REPRESENTATION COMPARISON")
print("="*45)
print(f"Rotation Matrix  : 9 parameters, no singularities, redundant")
print(f"Euler Angles     : 3 parameters, HAS singularities (gimbal lock)")
print(f"Quaternions      : 4 parameters, no singularities, efficient")
print()
print("Our quaternion [w,x,y,z]:", np.round(q_normal, 4))
print()
print("In ROS2, every transform uses quaternions.")
print("In Isaac Sim, every joint uses quaternions.")
print("Now i know exactly why.")

ORIENTATION REPRESENTATION COMPARISON
Rotation Matrix  : 9 parameters, no singularities, redundant
Euler Angles     : 3 parameters, HAS singularities (gimbal lock)
Quaternions      : 4 parameters, no singularities, efficient

Our quaternion [w,x,y,z]: [0.8224 0.3604 0.4397 0.0223]

In ROS2, every transform uses quaternions.
In Isaac Sim, every joint uses quaternions.
Now i know exactly why.
